In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('cleaned_data.csv')

In [4]:
#divide into X and y
X = df.drop(columns = ['Outcome'])
y = df[['Outcome']]

In [5]:
#feature scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = pd.DataFrame(data = sc.fit_transform(X), columns = X.columns)

In [6]:
#divide into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0, stratify = y)

In [7]:
print(y_train['Outcome'].value_counts())
print(y_test['Outcome'].value_counts())

Outcome
0    450
1    241
Name: count, dtype: int64
Outcome
0    50
1    27
Name: count, dtype: int64


In [8]:
#Lets create balanced data in training using SMOTE
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state = 42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [9]:
y_resampled['Outcome'].value_counts()

Outcome
0    450
1    450
Name: count, dtype: int64

In [10]:
#build model and train - LogisticRegression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['Outcome'], result['Prediction'])
score = accuracy_score(result['Outcome'], result['Prediction'])
print(cm)
print(score)

[[41  9]
 [ 6 21]]
0.8051948051948052


In [11]:
#build and train model - DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
model_dt = DecisionTreeClassifier(criterion = 'gini')
model_dt.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model_dt.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['Outcome'], result['Prediction'])
score = accuracy_score(result['Outcome'], result['Prediction'])
print(cm)
print(score)

[[45  5]
 [10 17]]
0.8051948051948052


In [12]:
#build model and train - RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(n_estimators = 101)
model_rf.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model_rf.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['Outcome'], result['Prediction'])
score = accuracy_score(result['Outcome'], result['Prediction'])
print(cm)
print(score)

[[46  4]
 [ 7 20]]
0.8571428571428571


In [13]:
#build and train model - KNN
from sklearn.neighbors import KNeighborsClassifier
model_knn = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
model_knn.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model_knn.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['Outcome'], result['Prediction'])
score = accuracy_score(result['Outcome'], result['Prediction'])
print(cm)
print(score)

[[38 12]
 [ 7 20]]
0.7532467532467533


In [14]:
#build model and train - GaussianNB
from sklearn.naive_bayes import GaussianNB
model_nb = GaussianNB()
model_nb.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model_nb.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['Outcome'], result['Prediction'])
score = accuracy_score(result['Outcome'], result['Prediction'])
print(cm)
print(score)

[[40 10]
 [ 9 18]]
0.7532467532467533


In [15]:
#build model and train - SVC
from sklearn.svm import SVC
model_svc = SVC(kernel = 'rbf')
model_svc.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model_svc.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['Outcome'], result['Prediction'])
score = accuracy_score(result['Outcome'], result['Prediction'])
print(cm)
print(score)

[[43  7]
 [ 6 21]]
0.8311688311688312


In [16]:
#accuracy score of random forest algo is large so choose to export this model
import pickle
pickle.dump(model_rf, open('model.pkl', 'wb'))
pickle.dump(sc, open('sc.pkl', 'wb'))

In [17]:
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
myinput = [[10, 150, 80, 35, 130, 30, 0.65, 55]]
myinput = pd.DataFrame(data  = myinput, columns = columns)
myinput = pd.DataFrame(data  = sc.transform(myinput), columns = columns)
result = model_rf.predict(myinput)
if result[0] == 0:
    print("No, patient is not diabetic")
else:
    print("Yes, patient is diabetic")

Yes, patient is diabetic
